In [ ]:
# ==============================================================================
# SCRIPT PRINCIPAL: FILTRADO GEOGRÁFICO Y CÁLCULO TOXICOLÓGICO (MÉTODO L.B. GRATT)
# ==============================================================================
# NOTA PARA EL COMITÉ EVALUADOR:
# Este algoritmo automatiza la depuración de la base de datos nacional del RETC 2023,
# aísla exclusivamente las emisiones de la Zona Metropolitana de Guadalajara (ZMG)
# y aplica las fórmulas matemáticas de L.B. Gratt para obtener los valores de riesgo.

import os
import re
from pathlib import Path
from typing import Dict, Any, Optional, Tuple
import pandas as pd # Librería 'pandas': Funciona como un motor de Excel interno para manipular tablas masivas rápidamente.

# ---------------------------------------------------------------------------
# 1. CONSTANTES GLOBALES Y PARÁMETROS GEOGRÁFICOS
# ---------------------------------------------------------------------------

# Factor de conversión: La metodología original de Gratt utiliza el sistema inglés.
KG_TO_LB: float = 2.20462262185

# Diccionario geográfico: Se enlistan los 9 municipios de la ZMG con sus
# posibles variaciones ortográficas (con y sin acentos) para evitar que el sistema
# descarte empresas válidas solo porque el capturista escribió "TONALA" en lugar de "TONALÁ".
GUADALAJARA_MUNICIPALITIES = {
    'GUADALAJARA',
    'ZAPOPAN',
    'SAN PEDRO TLAQUEPAQUE', 'TLAQUEPAQUE',
    'TONALA', 'TONALÁ',
    'TLAJOMULCO DE ZUÑIGA', 'TLAJOMULCO DE ZÚÑIGA',
    'EL SALTO',
    'JUANACATLAN', 'JUANACATLÁN',
    'IXTLAHUACAN DE LOS MEMBRILLOS', 'IXTLAHUACÁN DE LOS MEMBRILLOS',
    'ZAPOTLANEJO'
}

JALISCO_PATTERNS = [
    'JALISCO',
    'JAL',
]

# ---------------------------------------------------------------------------
# 2. FUNCIONES DE EXTRACCIÓN Y LIMPIEZA DE DATOS
# ---------------------------------------------------------------------------

# Función: load_iur_rel
# Propósito: Extrae el Factor de Riesgo Unitario (IUR/URF) y el Nivel de Referencia (REL).
def load_iur_rel(contable_path: Path) -> Dict[str, Dict[str, Any]]:
    data_by_cas: Dict[str, Dict[str, Any]] = {}

    # Lee el archivo de Excel 'contable.xlsx' (Base de la OEHHA).
    dfc = pd.read_excel(contable_path, sheet_name=0, header=0)
    cols = [str(c).upper() for c in dfc.columns]

    cas_col, iur_col, rel_col, name_col = None, None, None, None

    # BÚSQUEDA DINÁMICA DE COLUMNAS: En lugar de decirle al código "busca en la columna B",
    # busca palabras clave en los títulos. Así, si la OEHHA cambia el formato de su Excel
    # el próximo año, el código no se rompe y encuentra los datos automáticamente.
    for idx, col_name in enumerate(cols):
        if 'CHEMICAL' in col_name and 'ABSTRACT' in col_name: cas_col = dfc.columns[idx]
        if 'INHALATION' in col_name and 'UNIT' in col_name and 'RISK' in col_name: iur_col = dfc.columns[idx]
        if 'CHRONIC' in col_name and 'INHALATION' in col_name: rel_col = dfc.columns[idx]
        if 'SUBSTANCE' in col_name: name_col = dfc.columns[idx]

    if cas_col is None:
        for idx, col_name in enumerate(cols):
            if 'CAS' in col_name: cas_col = dfc.columns[idx]

    # Recorre fila por fila el documento. 'pd.notna' verifica que la celda no esté vacía.
    for row_idx, row in dfc.iterrows():
        cas = str(row[cas_col]).strip() if cas_col is not None and pd.notna(row[cas_col]) else None
        if not cas: continue # Si no hay número CAS, salta a la siguiente fila.

        # 'try/except' intenta convertir el valor a número decimal (float).
        # Si la celda tiene texto por error, la ignora para que no colapse el programa.
        iur = None
        if iur_col is not None and pd.notna(row[iur_col]):
            try: iur = float(row[iur_col])
            except (ValueError, TypeError): pass

        rel = None
        if rel_col is not None and pd.notna(row[rel_col]):
            try: rel = float(row[rel_col])
            except (ValueError, TypeError): pass

        name = str(row[name_col]).strip() if name_col is not None and pd.notna(row[name_col]) else None

        # Guarda los valores toxicológicos asociados al número CAS del químico.
        data_by_cas[cas] = {'iur': iur, 'rel': rel, 'name': name}

    return data_by_cas


# Función: _find_utm_columns
# Propósito: Busca dinámicamente las columnas de coordenadas UTM X y UTM Y.
def _find_utm_columns(df: pd.DataFrame) -> Tuple[Optional[str], Optional[str]]:
    utmx, utmy = None, None
    for col in df.columns:
        col_upper = str(col).upper()
        if 'UTM' in col_upper and 'X' in col_upper: utmx = col
        elif 'UTM' in col_upper and 'Y' in col_upper: utmy = col

    # Respaldo: Si no dice "UTM", asume por defecto las posiciones 9 y 10 típicas del RETC.
    if utmx is None and len(df.columns) > 9: utmx = df.columns[9]
    if utmy is None and len(df.columns) > 10: utmy = df.columns[10]
    return utmx, utmy


# Función: _load_retc_sheets
# Propósito: Carga las pestañas específicas del Excel gigante de Semarnat.
def _load_retc_sheets(retc_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # 'header=9' indica que se salte las primeras 9 filas del Excel,
    # ya que Semarnat suele poner logotipos y títulos antes de la tabla real.
    datos_generales = pd.read_excel(retc_path, sheet_name='Datos Generales', header=9)
    emisiones = pd.read_excel(retc_path, sheet_name='Emisiones y Transferencias', header=9)
    return datos_generales, emisiones


# Función: _build_emissions_table
# Propósito: Limpia la tabla de emisiones, dejando solo lo indispensable.
def _build_emissions_table(emisiones: pd.DataFrame) -> pd.DataFrame:
    # Se extraen solo 4 columnas clave: Número de Registro Ambiental (NRA), CAS, Sustancia y Emisión al AIRE.
    df = emisiones[['NRA', 'CAS', 'SUSTANCIA', 'AIRE']].copy()

    # Renombra la columna 'AIRE' por claridad.
    df.rename(columns={'AIRE': 'EMISION_KG_ANIO'}, inplace=True)

    # 'strip()' borra espacios en blanco accidentales que el capturista pudo dejar antes o después del texto.
    df['CAS'] = df['CAS'].astype(str).str.strip()

    # Filtra y elimina filas donde el CAS esté vacío o diga 'nan' (Not a Number).
    df = df[~df['CAS'].str.lower().isin(['nan', 'none', ''])]

    # Convierte las emisiones a valores numéricos. Si hay errores, pone 0 ('fillna(0)').
    df['EMISION_KG_ANIO'] = pd.to_numeric(df['EMISION_KG_ANIO'], errors='coerce').fillna(0)
    return df


# Función: _build_company_table
# Propósito: Limpia y prepara la tabla de datos de las empresas (ubicación y nombre).
def _build_company_table(datos_generales: pd.DataFrame, utmx_col: Optional[str], utmy_col: Optional[str]) -> pd.DataFrame:
    col_estado = 'ESTADO' if 'ESTADO' in datos_generales.columns else 'ENTIDAD'
    col_lat = next((c for c in datos_generales.columns if 'LATIT' in str(c).upper()), None)
    col_lon = next((c for c in datos_generales.columns if 'LONGIT' in str(c).upper()), None)

    base_cols = ['NRA', 'NOMBRE', col_estado, 'MUNICIPIO', col_lat, col_lon]
    if utmx_col is not None: base_cols.append(utmx_col)
    if utmy_col is not None: base_cols.append(utmy_col)

    dg = datos_generales[base_cols].copy()
    rename_map = {col_estado: 'ESTADO', col_lat: 'LATITUD', col_lon: 'LONGITUD'}
    if utmx_col is not None: rename_map[utmx_col] = 'UTMX'
    if utmy_col is not None: rename_map[utmy_col] = 'UTMY'
    dg.rename(columns=rename_map, inplace=True)

    # Convierte las coordenadas UTM de metros a kilómetros dividiendo entre 1000.
    dg['UTMX_KM'] = pd.to_numeric(dg['UTMX'], errors='coerce') / 1000.0 if 'UTMX' in dg.columns else None
    dg['UTMY_KM'] = pd.to_numeric(dg['UTMY'], errors='coerce') / 1000.0 if 'UTMY' in dg.columns else None
    return dg


# ---------------------------------------------------------------------------
# 3. FILTRADO ESPACIAL (ZMG)
# ---------------------------------------------------------------------------
def _filter_jalisco(merged: pd.DataFrame) -> pd.DataFrame:
    # 'str.replace' elimina puntos y espacios para que "JALISCO." o " JALISCO " sean idénticos al analizarlos.
    merged['ESTADO_NORM'] = (
        merged['ESTADO'].astype(str).str.upper().str.strip()
        .str.replace('.', '', regex=False).str.replace(' ', '', regex=False)
    )

    # Filtra primero a nivel estatal.
    mask = (
        merged['ESTADO_NORM'].isin(JALISCO_PATTERNS) |
        merged['ESTADO_NORM'].str.contains(r'JALISCO', na=False, regex=True)
    )
    jal = merged[mask].copy()
    jal.loc[:, 'MUNICIPIO_NORM'] = jal['MUNICIPIO'].astype(str).str.upper().str.strip()

    # 'isin' revisa si el municipio está dentro de la lista de la ZMG definida al inicio.
    jal.loc[:, 'ES_ZM_GUADALAJARA'] = jal['MUNICIPIO_NORM'].isin(GUADALAJARA_MUNICIPALITIES)

    # FILTRO ESTRICTO: Conserva únicamente los registros marcados como True (Sí pertenecen a la ZMG).
    amg_only = jal[jal['ES_ZM_GUADALAJARA'] == True].copy()

    return amg_only


# ---------------------------------------------------------------------------
# 4. MOTOR MATEMÁTICO: METODOLOGÍA L.B. GRATT Y RIESGO TOXICOLÓGICO
# ---------------------------------------------------------------------------
def _calculate_risk_scores(pivot: pd.DataFrame, toxics: pd.DataFrame, contable_data: Dict[str, Dict[str, Any]], dilution_factor: float, dilution_factor2: float) -> Tuple[pd.DataFrame, pd.DataFrame, list]:
    id_cols = ['NRA', 'NOMBRE', 'ESTADO', 'MUNICIPIO', 'LATITUD', 'LONGITUD', 'UTMX_KM', 'UTMY_KM']
    pivot_noncancer = pivot[id_cols].copy()
    cancer_cols_dict: Dict[str, Any] = {}
    noncancer_cols_dict: Dict[str, Any] = {}
    substances_without_rel: list = []

    # Bucle que evalúa químco por químico (usando su número CAS).
    for cas in toxics['CAS'].unique():
        entry = contable_data.get(cas, {})
        iur: float = entry.get('iur') or 0.0
        rel: Optional[float] = entry.get('rel')

        # Si la industria emitió este químico, procede al cálculo.
        if cas in pivot.columns:
            emission_kg = pivot[cas].astype(float)

            # --- CÁLCULO DE RIESGO CARCINOGÉNICO (CÁNCER) ---
            # Se convierte la emisión anual de kg a lb, se divide entre 365 días y se multiplica
            # por el factor de dilución atmosférica para estimar la concentración real en el aire.
            conc_m1 = emission_kg * KG_TO_LB / 365.0 * dilution_factor
            cancer_cols_dict[f'CONC_ugm3_{cas}'] = conc_m1

            # Riesgo Final = Concentración calculada * Factor Unitario de Riesgo (IUR)
            cancer_cols_dict[f'RISK_{cas}'] = conc_m1 * iur

            # Se aplica el segundo factor de dilución (dilution_factor2) para cálculos comparativos adicionales.
            conc_m2 = emission_kg * KG_TO_LB * 7700.0 * dilution_factor2
            cancer_cols_dict[f'CONC_SCORE_{cas}'] = conc_m2
            cancer_cols_dict[f'CANCER_SCORE_{cas}'] = conc_m2 * iur

            # --- CÁLCULO DE RIESGO CRÓNICO NO CARCINOGÉNICO (Cociente de Peligro - HQ) ---
            emission_lb_yr = emission_kg * KG_TO_LB

            # Solo si el químico tiene un valor REL reportado por la OEHHA, calcula el riesgo crónico.
            if rel is not None and rel > 0:
                # HQ = Concentración / REL. (Valores mayores a 1.0 exceden la norma de seguridad).
                noncancer_cols_dict[f'NONCANCER_SCORE_{cas}'] = emission_lb_yr * dilution_factor2 / rel
            else:
                # 'pd.Series([float('nan')])' inserta un valor "Nulo" para químicos sin REL, evitando alterar promedios.
                noncancer_cols_dict[f'NONCANCER_SCORE_{cas}'] = pd.Series([float('nan')] * len(pivot), index=pivot.index)
                subst_name = entry.get('name', toxics.loc[toxics['CAS'] == cas, 'SUSTANCIA'].iloc[0] if cas in toxics['CAS'].values else 'Desconocida')
                substances_without_rel.append(f"{cas} ({subst_name})")

        # Si la industria no emitió este químico, el riesgo es 0.
        else:
            cancer_cols_dict[f'CONC_ugm3_{cas}'] = 0.0
            cancer_cols_dict[f'RISK_{cas}'] = 0.0
            cancer_cols_dict[f'CONC_SCORE_{cas}'] = 0.0
            cancer_cols_dict[f'CANCER_SCORE_{cas}'] = 0.0
            noncancer_cols_dict[f'NONCANCER_SCORE_{cas}'] = 0.0

    # Se pegan (concat) las columnas de resultados nuevos a la tabla principal.
    cancer_new = pd.DataFrame(cancer_cols_dict, index=pivot.index)
    pivot = pd.concat([pivot, cancer_new], axis=1)

    noncancer_new = pd.DataFrame(noncancer_cols_dict, index=pivot_noncancer.index)
    pivot_noncancer = pd.concat([pivot_noncancer, noncancer_new], axis=1)

    # 'sum(axis=1)' suma el riesgo individual de cada químico emitido por la empresa
    # para obtener el Riesgo Total de esa instalación industrial.
    pivot['RISK_TOTAL'] = pivot[[c for c in pivot.columns if c.startswith('RISK_')]].sum(axis=1)
    pivot['CANCER_SCORE_TOTAL'] = pivot[[c for c in pivot.columns if c.startswith('CANCER_SCORE_')]].sum(axis=1)
    pivot_noncancer['NONCANCER_SCORE_TOTAL'] = pivot_noncancer[[c for c in pivot_noncancer.columns if c.startswith('NONCANCER_SCORE_')]].sum(axis=1, skipna=True)

    return pivot, pivot_noncancer, substances_without_rel


# ---------------------------------------------------------------------------
# 5. ORQUESTADOR Y EXPORTACIÓN DE RESULTADOS
# ---------------------------------------------------------------------------
def process_file(retc_path: Path, contable_data: Dict[str, Dict[str, Any]], output_dir: str, dilution_factor: float = 2.47, dilution_factor2: float = 0.25) -> Path:
    # 1. Se cargan y limpian las tablas.
    datos_generales, emisiones = _load_retc_sheets(retc_path)
    utmx_col, utmy_col = _find_utm_columns(datos_generales)
    company_table = _build_company_table(datos_generales, utmx_col, utmy_col)
    emissions_table = _build_emissions_table(emisiones)

    # 2. 'merge' funciona exactamente igual a un BUSCARV en Excel:
    # Une la tabla de emisiones con los datos geográficos de la empresa utilizando su número NRA.
    merged = emissions_table.merge(company_table, on='NRA', how='left')

    # 3. Se aplica el filtro para dejar únicamente la ZMG.
    merged_jal = _filter_jalisco(merged)

    # 'drop_duplicates' elimina filas repetidas para generar catálogos únicos de empresas y sustancias.
    toxics = merged_jal[['CAS', 'SUSTANCIA']].drop_duplicates().reset_index(drop=True)
    companies = company_table.merge(merged_jal[['NRA']].drop_duplicates(), on='NRA', how='inner').drop_duplicates().reset_index(drop=True)

    # 4. 'pivot_table' funciona como una Tabla Dinámica de Excel:
    # Agrupa a las empresas por fila y convierte las sustancias (CAS) en columnas individuales.
    pivot_index = ['NRA', 'NOMBRE', 'ESTADO', 'MUNICIPIO', 'LATITUD', 'LONGITUD', 'UTMX_KM', 'UTMY_KM']
    pivot = merged_jal.pivot_table(index=pivot_index, columns='CAS', values='EMISION_KG_ANIO', aggfunc='sum', fill_value=0).reset_index()

    # 5. Se ejecuta el motor matemático de riesgos.
    pivot, pivot_noncancer, substances_without_rel = _calculate_risk_scores(pivot, toxics, contable_data, dilution_factor, dilution_factor2)

    # 6. EXPORTACIÓN: Crea una carpeta con el año (ej. output_2023) y guarda todo en archivos .csv
    year_match = re.search(r'(\d{4})', retc_path.name)
    year = year_match.group(1) if year_match else retc_path.stem
    out_folder = Path(output_dir) / f'output_{year}'
    out_folder.mkdir(parents=True, exist_ok=True)

    # 'to_csv' convierte las tablas internas de Python a archivos leíbles en Excel.
    companies.to_csv(out_folder / f'{year}_companies.csv', index=False, encoding='utf-8')
    toxics.to_csv(out_folder / f'{year}_toxics.csv', index=False, encoding='utf-8')
    pivot.to_csv(out_folder / f'{year}_company_emissions_risk.csv', index=False, encoding='utf-8')
    pivot_noncancer.to_csv(out_folder / f'{year}_company_noncancer_score.csv', index=False, encoding='utf-8')

    print(f"  - Registros en Jalisco       : {len(merged_jal)}")
    print(f"  - Empresas únicas            : {len(companies)}")
    return out_folder

# ---------------------------------------------------------------------------
# 6. BLOQUE DE EJECUCIÓN EN GOOGLE COLAB
# ---------------------------------------------------------------------------
INPUT_DIR = '/content/'          # Directorio raíz de Colab donde se suben los Excels.
OUTPUT_DIR = '/content/resultados' # Directorio donde el programa guardará los .csv finales.

DILUTION_FACTOR = 2.47
DILUTION_FACTOR_2 = 0.25

input_path = Path(INPUT_DIR)
contable_path = input_path / 'contable.xlsx'

if not contable_path.exists():
    print("Por favor sube el archivo 'contable.xlsx' a Colab antes de ejecutar.")
else:
    print("Cargando valores IUR y REL desde contable.xlsx...")
    contable_data = load_iur_rel(contable_path)

    # 'iterdir' busca automáticamente dentro de la carpeta todos los archivos
    # que se llamen "retc" y terminen en formato de Excel (.xlsx o .xls).
    retc_files = [
        f for f in input_path.iterdir()
        if f.name.lower().startswith('retc') and f.suffix.lower() in ('.xlsx', '.xls')
    ]

    if not retc_files:
        print("No se encontraron archivos que empiecen con 'retc' en la carpeta.")
    else:
        # Bucle final: Manda a procesar cada archivo RETC encontrado.
        for rf in retc_files:
            print(f"Procesando {rf.name}...")
            try:
                out_path = process_file(rf, contable_data, OUTPUT_DIR, DILUTION_FACTOR, DILUTION_FACTOR_2)
                print(f"  ✓ Resultados guardados en: {out_path}\n")
            except Exception as exc:
                print(f"  ✗ Error procesando {rf.name}: {exc}\n")